In [ ]:


import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
import sys
sys.path.append(str(Path.home()/'Documents'/'Projects'/'Regional-Monitoring'/'Indicator_Gen'/'config'))
import functions as func

pd.set_option('display.max_columns', None)

PATH_SP = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Products' / 'SACOG Peer Regions tours' / '2026' / '2026 peer region tour data outputs' / '2026 update'
PATH_I = Path(r'I:\Projects\Josh\Regional Monitoring')

def re_remove_post(x, exp = ' '):
    try:
        return x.split(exp, 1)[0]
    except Exception as e:
        e
        return x



In [ ]:


EXPORT=True

file_in=PATH_SP/'MnR_outputs'/'Cost_6 MSA PUMS5_og.xlsx'
df = pd.read_excel(file_in, sheet_name='PUMA')

print(df['MSA'].unique())
display(df.head(2))
display(df.tail(2))

def wm(x):
    return np.average(x, weights = df.loc[x.index, 'Households'])
def sqrtsumsq(x):
    return np.sqrt(np.sum(x**2))

df.loc[df['MSA'].str.contains('Sacramento|Yuba'), 'MSA'] = 'SACOG'
df['MSA_ID'] = df['MSA_ID'].astype(str)
df.loc[df['MSA'] == 'SACOG', 'MSA_ID'] = '40900, 49700'

df['Households'] = df['Households'].replace(0, 1)
df = df.groupby(['MSA_ID', 'MSA', 'Year', 'RAC1P', 'Housing Type', 'Housing Burden'], as_index=False).agg(Households=('Households', 'sum'), Percent=('Percent', wm), ME=('Margin of Error', sqrtsumsq))
df['Margin of Error Ratio'] = df['ME']/df['Households']

conditions = [
    df['Margin of Error Ratio'] <= 0.05
    , df['Margin of Error Ratio'] > 0.05
]

choices = ['Yes', 'No']

df['Use for Reporting?'] = np.select(conditions, choices, default='no')
df['Households'] = df['Households'].replace(1, 0)

factor_burden = ['Housing data not available', 'N/A (GQ/vacant/not owned or being bought/occupied without rent payment/no household income)', 'Cost burden <=30%', 'Cost burden >30% to <=50%', 'Cost burden >50%']
df['housing_burden_sort'] = pd.Categorical(df['Housing Burden'], factor_burden)

factor_types = ['Housing data not available', 'N/A (GQ/vacant/not owned or being bought/occupied without rent payment/no household income)', 'Owner', 'Renter', 'Owners and Renters']
df['housing_type_sort'] = pd.Categorical(df['Housing Type'], factor_types)

factor_race = ['All', 'American Indian or Alaska Native (NH)', 'Asian (NH)', 'Black or African American (NH)', 'Hispanic or Latino', 'Native Hawaiian or other Pacific Islander (NH)', 'White (NH)', 'Some other race (NH)', 'Two or more races (NH)']
df['RAC1P_sort'] = pd.Categorical(df['RAC1P'], factor_race)

df = df.sort_values(by= ['MSA_ID', 'Year', 'RAC1P_sort', 'housing_type_sort', 'housing_burden_sort'], ascending=[True, False, True, True, True])
df = df.drop(['RAC1P_sort', 'housing_type_sort', 'housing_burden_sort'], axis=1)
df = df.reset_index(drop=True)
display(df.head())


if EXPORT:

    estimate = 'ACS5'
    sample_type = 'PUMS'
    indicator = 'Cost_6'
    start_year = int(df['Year'].min())
    end_year = int(df['Year'].max())
    geography = 'MSA'

    params = {
        'estimate': estimate
        , 'sample': sample_type
        , 'indicator': indicator
        , 'start_year': start_year
        , 'end_year': end_year
        , 'geo': geography
        , 'moe_thresh': None
    }

    # Create about documentation page for export
    df_about = func.write_about(params)

    print("Visual representation of the output for:", indicator)
    display(df_about)

    file_out = PATH_SP/'Cost_6 MSA PUMS5_prt_2026.xlsx'
    with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
        df_about.to_excel(writer, index=False, sheet_name='About', header=False)
        df      .to_excel(writer, index=False, sheet_name='MSA'                )
